ROC-
AUC-AREA UNDER THE CURVE  X-AXIS-FLASE+VE Y-AXIS- -VE      
  UPPER MA CURVE-MODEL WORK GOOD     
  MIDDLE DEKHI TALA VAYA MISSCALSSIFICATION BUJAUXA

In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch_geometric.transforms as T
from torch_geometric.datasets import Planetoid
from torch_geometric.nn import GCNConv
from torch_geometric.utils import negative_sampling
from sklearn.metrics import roc_auc_score,average_precision_score
device=torch.device("cuda" if torch.cuda.is_available()else "cpu")
print("device",device)

device cpu


edge index edgelabelindex-inf kun souce and destination kun ho ysma inf store hunxa


Output

	

Meaning




x=[2708, 1433]

	

2,708 research papers, each with 1,433 features.




edge_index=[2, 10556]

	

10,556 stored edge entries representing citation connections.




y=[2708]

	

One node-class label per paper, if labels are included.




Nodes: 2708

	

Total number of papers.




Edges: 10556

	

Number of stored edge entries, not necessarily the number of unique undirected connections.




Features: torch.Size([2708, 1433])

	

The shape of the feature matrix.

In [5]:
import torch
data=torch.load("cora_graph.pt",map_location="cpu",weights_only=False)
print(data)
print("Nodes:", data.num_nodes)
print("Edges:",data.edge_index.shape[1])
print("Features:",data.x.shape)

Data(x=[2708, 1433], edge_index=[2, 10556], y=[2708], train_mask=[2708], val_mask=[2708], test_mask=[2708])
Nodes: 2708
Edges: 10556
Features: torch.Size([2708, 1433])


row=2708
features=col=1433 node
total edges=10556
edge_index=[2, 8976]

In [6]:
split_transform=T.RandomLinkSplit(num_val=0.05,num_test=0.1,is_undirected=True, add_negative_train_samples=False)
train_data,val_data,test_data=split_transform(data)
print(train_data)
print(train_data.edge_label_index[0])
print(train_data.edge_label_index[1])

Data(x=[2708, 1433], edge_index=[2, 8976], y=[2708], train_mask=[2708], val_mask=[2708], test_mask=[2708], edge_label=[4488], edge_label_index=[2, 4488])
tensor([180, 961, 978,  ..., 205, 869, 924])
tensor([2076, 2599,  998,  ..., 1869, 1440, 2383])


In [7]:
print(test_data)

Data(x=[2708, 1433], edge_index=[2, 9502], y=[2708], train_mask=[2708], val_mask=[2708], test_mask=[2708], edge_label=[1054], edge_label_index=[2, 1054])


In [8]:
print(val_data)

Data(x=[2708, 1433], edge_index=[2, 8976], y=[2708], train_mask=[2708], val_mask=[2708], test_mask=[2708], edge_label=[526], edge_label_index=[2, 526])


In [10]:
train_data=train_data.to(device)
val_data=val_data.to(device)
test_data=test_data.to(device)
all_original_edges=data.edge_index.to(device)


drop- k hi % le ghatauxa

In [11]:

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cpu


In [12]:
class GCNlink(nn.Module):

    def __init__(self):
        super().__init__()
        self.conv1=GCNConv(1433,128)
        self.conv2=GCNConv(128,64)
        
    def encode(self,x,edge_index):
        x=self.conv1(x,edge_index)
        x=F.relu(x)
        x=F.dropout(x,p=0.3)
        z=self.conv2(x,edge_index)
        return z
    def decode(self,z,edges_label_index):
        source=edges_label_index[0]
        destination=edges_label_index[1]
        scores=(z[source]*z[destination]).sum(dim=-1)
        return scores

todevice- jaha model load vaxa ni tyhi load grna like jaha train garinxa tyhi rahos vnyara

In [13]:
model=GCNlink().to(device)
optimizer=torch.optim.Adam(model.parameters(),lr=0.01)
criterion =nn.BCEWithLogitsLoss()
print(model)

GCNlink(
  (conv1): GCNConv(1433, 128)
  (conv2): GCNConv(128, 64)
)


In [17]:
def train():
    model.train()
    optimizer.zero_grad()
#generating embedding for training dataset
    z=model.encode(train_data.x,train_data.edge_index)
    pos_edge_index =train_data.edge_label_index
    pos_scores=model.decode(z,pos_edge_index)
#virtually connecting each node as given

    neg_edge_index=negative_sampling(
        edge_index=all_original_edges,
        num_nodes=data.num_nodes,
        num_neg_samples=pos_edge_index.size(1),
        method="sparse"
    )

    neg_scores=model.decode(z,neg_edge_index)
    scores=torch.cat([pos_scores,neg_scores])
    labels=torch.cat([torch.ones_like(pos_scores),
                      torch.zeros_like(neg_scores)
                    ])
    loss=criterion(scores,labels)
    loss.backward()
    optimizer.step()
    return loss.item()



In [18]:
@torch.no_grad()
def evaluate(split_data):
    model.eval()
    z=model.encode(split_data.x,split_data.edge_index)
    scores=model.decode(z,split_data.edge_label_index)
    probabilities=torch.sigmoid(scores)
    labels=split_data.edge_label
    y_true=labels.cpu().numpy()
    y_score=probabilities.cpu().numpy()
    auc=roc_auc_score(y_true,y_score)
    ap=average_precision_score(y_true,y_score)
    return auc,ap

In [19]:
epochs=100
for epoch in range(1,epochs+1):
    loss=train()
    val_auc,val_ap=evaluate(val_data)
    if epoch%10 ==0:
        print(f"epoch{epoch:03d} loss:{loss:4f} val auc:{val_auc}")
test_auc,test_ap=evaluate(test_data)
print("\n final test results")
print(f"test roc-auc:{test_auc:4f}")
print(f"test avg percision:{test_ap:.4f}")

epoch010 loss:0.687672 val auc:0.6780349578568434
epoch020 loss:0.630964 val auc:0.7094218508291287
epoch030 loss:0.557906 val auc:0.7469386574910727
epoch040 loss:0.514665 val auc:0.8196157238069077
epoch050 loss:0.497989 val auc:0.8273359452934117
epoch060 loss:0.477651 val auc:0.8669779814656854
epoch070 loss:0.461922 val auc:0.8996082059882318
epoch080 loss:0.456406 val auc:0.898813052089809
epoch090 loss:0.453083 val auc:0.9075452876288511
epoch100 loss:0.437553 val auc:0.9042200985990834

 final test results
test roc-auc:0.896050
test avg percision:0.8923
